## 1. Crear base de datos y cargar datos

In [1]:
import sqlite3
from pathlib import Path

conn = sqlite3.connect("db.sqlite")
cur = conn.cursor()

for fname in ["schema.sql", "grados_familias_ciclos_modulos.sql"]:
    sql = Path(fname).read_text(encoding="utf-8")
    cur.executescript(sql)

conn.commit()
print("Base de datos cargada correctamente.")

Base de datos cargada correctamente.


## 2. Módulos por ciclo formativo (con familia y grado)

In [2]:
import pandas as pd

query = """
SELECT
    g.nombre      AS grado,
    f.codigo      AS cod_familia,
    f.nombre      AS familia,
    c.nombre      AS ciclo,
    m.id_oficial  AS cod_modulo,
    m.nombre      AS modulo
FROM modulos m
JOIN ciclos   c ON m.id_ciclo  = c.id
JOIN familias f ON c.id_familia = f.id
JOIN grados   g ON c.id_grado   = g.id
ORDER BY g.id, f.codigo, c.nombre, m.id_oficial;
"""

df = pd.read_sql_query(query, conn)
print(f"Total filas: {len(df)}")


Total filas: 5061


## 3. Filtrar por ciclo concreto

In [4]:
# Cambia este valor para filtrar por cualquier ciclo
CICLO = "Técnico en Gestión Administrativa"

df_ciclo = df[df["ciclo"] == CICLO].reset_index(drop=True)
print(f"Ciclo: {CICLO}")
print(f"Familia: {df_ciclo['familia'].iloc[0]}  |  Grado: {df_ciclo['grado'].iloc[0]}")
print(f"Nº módulos: {len(df_ciclo)}")
df_ciclo[["cod_modulo", "modulo"]]

Ciclo: Técnico en Gestión Administrativa
Familia: Administración y gestión  |  Grado: Grado Medio
Nº módulos: 12


,cod_modulo,modulo
0,0156,Inglés
1,0437,Comunicación empresarial y atención al cliente
2,0438,Operaciones administrativas de compra-venta
3,0439,Empresa y Administración
4,0440,Tratamiento informático de la información
5,0441,Técnica contable
6,0442,Operaciones administrativas de recursos humanos
7,0443,Tratamiento de la documentación contable
8,0446,Empresa en el aula
9,0448,Operaciones auxiliares de gestión de tesorería


## 4. Resumen: número de módulos por ciclo

In [5]:
resumen = (
    df.groupby(["grado", "cod_familia", "familia", "ciclo"])
    .size()
    .reset_index(name="num_modulos")
)
resumen

,grado,cod_familia,familia,ciclo,num_modulos
0,Básica,ADG,Administración y gestión,Profesional Básico en Informática de Oficina,24
1,Básica,ADG,Administración y gestión,Profesional Básico en Servicios Administrativos,24
2,Básica,AFD,Actividades físicas y deportivas,Profesional Básico en Acceso y Conservación en...,14
3,Básica,AGA,Agraria,Profesional Básico en Actividades Agropecuarias,28
4,Básica,AGA,Agraria,Profesional Básico en Agro-jardinería y Compos...,28
...,...,...,...,...,...
204,Grado Superior,TMV,Transporte y mantenimiento de vehículos,Técnico Superior en Mantenimiento Aeromecánico...,33
205,Grado Superior,TMV,Transporte y mantenimiento de vehículos,Técnico Superior en Mantenimiento Aeromecánico...,30
206,Grado Superior,TMV,Transporte y mantenimiento de vehículos,Técnico Superior en Mantenimiento Aeromecánico...,30
207,Grado Superior,TMV,Transporte y mantenimiento de vehículos,Técnico Superior en Mantenimiento de Sistemas ...,30


## 5. Cerrar conexión

In [6]:
conn.close()
print("Conexión cerrada.")

Conexión cerrada.
